In [ ]:
import pandas as pd

In [ ]:
# importação de dados

df_vendas1 = pd.read_csv("/content/drive/MyDrive/!Dados/base_dados_vendas_tratada.csv", parse_dates=['dt_venda','dt_entrega'])

In [ ]:
df_vendas2 = pd.read_csv("/content/drive/MyDrive/!Dados/dados_complementares_v2.csv", parse_dates=['dt_venda','dt_entrega'])

In [ ]:
df_metas = pd.read_excel("/content/drive/MyDrive/!Dados/Metas.xlsx")

In [ ]:
df_vendas1.head()

In [ ]:
df_vendas2.head()

In [ ]:
df_metas.head()

In [ ]:
df_vendas1.info()

In [ ]:
df_vendas2.info()

In [ ]:
df_metas.info()

In [ ]:
df_vendas1.isnull().sum()

In [ ]:
df_vendas2.isnull().sum()

In [ ]:
df_metas.isnull().sum()

In [ ]:
# valores estatísticos

df_vendas1[['preco_custo','valor_unitario','quantidade','valor_final']].describe().round(2)

In [ ]:
df_vendas2[['preco_custo','valor_unitario','quantidade','valor_final']].describe().round(2)

In [ ]:
df_metas['meta'].describe().round(2)

In [ ]:
# JUNÇÃO DOS DADOS
# CONCATENAR

df_vendas = pd.concat([df_vendas1, df_vendas2], axis=0, join='outer')

In [ ]:
df_vendas.info()

In [ ]:
df_vendas.duplicated().sum()

In [ ]:
display(df_vendas)

In [ ]:
# MESCLAGEM

In [ ]:
df_vendas['mes'] = df_vendas['dt_venda'].dt.month
df_vendas['ano'] = df_vendas['dt_venda'].dt.year

In [ ]:
df_vendas.info()

In [ ]:
df_vendas.head(20)

In [ ]:
df_final = df_vendas.merge(df_metas, on=['mes','ano'], how='inner')

In [ ]:
df_final.info()

In [ ]:
df_final.head(20)

In [ ]:
df_final['lucro'] = (df_final['valor_unitario'] - df_final['preco_custo']) * df_final['quantidade']
df_final['margem_lucro'] = (df_final['valor_unitario'] - df_final['preco_custo']) / df_final['valor_unitario'] * 100
df_final['prazo_entrega'] = df_final['dt_entrega'] - df_final['dt_venda']
df_final['valor_final'] = df_final['valor_unitario'] * df_final['quantidade']

In [ ]:
df_final

In [ ]:
df_final.drop(columns=['matricula_funcionario','codigo_produto','codigo_loja'], inplace=True)

In [ ]:
df_final.info()

In [ ]:
# SALVANDO A BASE

df_final.to_csv("/content/drive/MyDrive/!Dados/base_final.csv")
df_final.to_excel("/content/drive/MyDrive/!Dados/base_final.xlsx")
df_final.to_json("/content/drive/MyDrive/!Dados/base_final.json")
df_final.to_xml("/content/drive/MyDrive/!Dados/base_final.xml")

In [ ]:
# ANÁLISE DESCRITIVA

# FILTRO LÓGICO DE DADOS

# > maior que
# < menor que
# >= maior ou igual que
# <= menor ou igual que
# == igual a
# != diferente de
# & (E)
# | (OU)
# ~ (NÃO)

In [ ]:
df_final.describe(include='object')

In [ ]:
df_final.nome_loja.unique()

In [ ]:
df_final.nome_funcionario.unique()

In [ ]:
df_final.descricao_produto.unique()

In [ ]:
# FILTROS

In [ ]:
filtro_loja = df_final['nome_loja'] == "Filial SP"
df_final[filtro_loja]

In [ ]:
filtro_fatura = df_final['valor_final'] > 12000
df_final[filtro_fatura]

In [ ]:
df_final[filtro_loja & filtro_fatura]

In [ ]:
df_final[filtro_loja | filtro_fatura]

In [ ]:
# QUERY - CONSULTA

In [ ]:
df_final.query("nome_funcionario == 'Rita Sampaio'")

In [ ]:
df_final.query("valor_final <= 1000")

In [ ]:
df_final.query("nome_funcionario == 'Jorge Mateus' & valor_final < 1500")

In [ ]:
# AGRUPAMENTOS

vendedores_vf = df_final.groupby('nome_funcionario')['valor_final'].sum().sort_values(ascending=False).reset_index()
display(vendedores_vf)

In [ ]:
vendedores_qtd = df_final.groupby('nome_funcionario')['quantidade'].sum().sort_values(ascending=False).reset_index()
display(vendedores_qtd)

In [ ]:
vendedores_vf_qtd = df_final.groupby('nome_funcionario')[['valor_final', 'quantidade']].sum().sort_values(by='valor_final', ascending=False).reset_index()
display(vendedores_vf_qtd)

In [ ]:
vendedores_prod_qtd = df_final.groupby(['nome_funcionario','descricao_produto'])['quantidade'].sum().sort_values(ascending=False).reset_index()
display(vendedores_prod_qtd)

In [ ]:
# GRÁFICOS

import plotly.express as px

In [ ]:
receitas_lojas = df_final.groupby('nome_loja')['valor_final'].sum().sort_values().reset_index()
display(receitas_lojas)

In [ ]:
fig = px.bar(receitas_lojas, x='nome_loja' , y='valor_final', title="Receitas por Loja", color='valor_final', text=receitas_lojas['valor_final'])

fig.update_traces(textposition='outside', texttemplate='%{text:.4s}')

fig.show()

In [ ]:
quantidade_prod = df_final.groupby('descricao_produto')['quantidade'].sum().sort_values().reset_index()
display(quantidade_prod)

In [ ]:
fig = px.bar(quantidade_prod, x='descricao_produto' , y='quantidade', title="Quantidade Vendida por Produto", color='quantidade', text=quantidade_prod['quantidade'])

fig.update_traces(textposition='outside')

fig.show()

In [ ]:
fat_lojas = df_final.groupby('nome_loja').agg({'valor_final':'sum', 'lucro':'sum'}).sort_values(by='valor_final', ascending=False).reset_index()

fig = px.bar(fat_lojas, x='nome_loja', y=['valor_final','lucro'], barmode='group', text=fat_lojas['valor_final'])

fig.update_traces(textposition='outside', texttemplate= '%{text:.2s}')

fig.show()

In [ ]:
categorias = df_final['categoria'].value_counts().reset_index().sort_values(by='count', ascending=False)
display(categorias)

,categoria,count
0,Casa,1050
1,Eletrodomésticos,923
2,Cozinha,839
3,Eletrônicos,688


In [ ]:
fig = px.pie(categorias, names='categoria', values='count', height=500, width=500)
fig.update_traces(hole=0.4, pull=[0.2])
fig.show()

In [ ]:
df_final['ano - mes'] = df_final['dt_venda'].dt.to_period('M')
df_fat_mes = df_final.groupby('ano - mes')['valor_final'].sum().reset_index()

df_fat_mes['ano - mes'] = df_fat_mes['ano - mes'].dt.to_timestamp()

fig = px.line(df_fat_mes, x='ano - mes', y='valor_final', line_shape='spline', markers=True)

fig.show()

In [ ]:
lucro_vf = df_final.groupby('descricao_produto').agg({
    'valor_final':'sum',
    'lucro':'sum',
    'quantidade':'sum'
}).reset_index()

fig = px.scatter(lucro_vf, x='lucro', y='valor_final', color='descricao_produto', size='quantidade', size_max=25)

fig.show()

In [ ]:
vendedores_geral = df_final.groupby(['nome_loja','nome_funcionario','descricao_produto'])['lucro'].sum().reset_index()
#display(vendedores_geral)
fig = px.treemap(vendedores_geral, path=[px.Constant('Geral'),'nome_loja','nome_funcionario','descricao_produto'], values='lucro')

fig.show()

In [ ]:
fig = px.sunburst(vendedores_geral, path=[px.Constant('Geral'),'nome_loja','nome_funcionario','descricao_produto'], values='lucro')

fig.show()

In [ ]:
Fabio
Paulo
Artur
Gabriele
Breno